In [7]:

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

folder = Path.cwd()
df = pd.read_csv(folder / "Housing.csv")

X = df.drop(columns=["price"])
y = df["price"]

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

def make_preprocessor(numeric_columns):
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer([
        ("num", numeric, numeric_columns),
        ("cat", categorical, cat_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

regression_model = Pipeline([
    ("preprocessing", make_preprocessor(num_cols)),
    ("model", LinearRegression())
])

regression_model.fit(X_train, y_train)
reg_predictions = regression_model.predict(X_test)

print("HOUSE PRICE REGRESSION")
print("MAE:", round(mean_absolute_error(y_test, reg_predictions), 2))
print("R²:", round(r2_score(y_test, reg_predictions), 3))

joblib.dump({
    "model": regression_model,
    "feature_columns": X.columns.tolist()
}, folder / "house_price_model.joblib")

train_estimates = cross_val_predict(
    Pipeline([
        ("preprocessing", make_preprocessor(num_cols)),
        ("model", LinearRegression())
    ]),
    X_train, y_train, cv=5
)

test_estimates = regression_model.predict(X_test)

def deal_category(asking_prices, estimated_prices):
    ratio = np.asarray(asking_prices) / np.maximum(estimated_prices, 1)

    return np.where(
        ratio < 0.90, 0,
        np.where(ratio <= 1.10, 1, 2)
    )

y_train_class = deal_category(y_train, train_estimates)
y_test_class = deal_category(y_test, test_estimates)

X_train_class = X_train.copy()
X_test_class = X_test.copy()

X_train_class["asking_price"] = y_train.values
X_test_class["asking_price"] = y_test.values

classification_model = Pipeline([
    ("preprocessing", make_preprocessor(num_cols + ["asking_price"])),
    ("model", LogisticRegression(max_iter=2000))
])

classification_model.fit(X_train_class, y_train_class)
class_predictions = classification_model.predict(X_test_class)

print("\nPROPERTY DEAL CLASSIFICATION")
print("Accuracy:",
      round(accuracy_score(y_test_class, class_predictions) * 100, 2), "%")

categories = {
    0: "Potential Bargain",
    1: "Fair Price",
    2: "Potentially Overpriced"
}

values, counts = np.unique(y_train_class, return_counts=True)

for value, count in zip(values, counts):
    print(categories[int(value)], ":", count)

joblib.dump({
    "model": classification_model,
    "feature_columns": X_train_class.columns.tolist()
}, folder / "classification_model.joblib")

print("\nBoth models saved successfully!")


HOUSE PRICE REGRESSION
MAE: 970043.4
R²: 0.653

PROPERTY DEAL CLASSIFICATION
Accuracy: 93.58 %
Potential Bargain : 137
Fair Price : 183
Potentially Overpriced : 116

Both models saved successfully!
